In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from typing import List, Tuple
import re
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer
from src.gpt import KilterGPT
import numpy as np
from datasets import disable_progress_bar
import wandb

disable_progress_bar()

# Initialize wandb
wandb.init(
    project="climb-gpt",  # Change this to your project name
    name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config={
        "architecture": "GPT",
        "n_embd": 256,
        "n_head": 4,
        "n_layer": 6,
        "n_positions": 128,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": 16,
        "learning_rate": 1e-5,
        "weight_decay": 0.01,
        "gradient_accumulation_steps": 1,
        "early_stopping_patience": 5,
    }
)

run_name = wandb.run.name  # Use wandb run name for consistency
OUT_DIR = f"models/climb_gpt/{run_name}"
device = "cuda" if torch.cuda.is_available() else "cpu"

dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80, 10, 10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

# Log dataset sizes
wandb.config.update({
    "train_size": len(datasets['train']),
    "val_size": len(datasets['val']),
    "test_size": len(datasets['test'])
})

tokenizer = train_tokenizer(datasets, OUT_DIR)
datasets = dp.preprocess_datasets(datasets, tokenizer)

# Log vocabulary size
wandb.config.update({"vocab_size": tokenizer.vocab_size})

model = KilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
)

# Watch model with wandb
wandb.watch(model.model, log="all", log_freq=1000)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,  # Log more frequently for wandb
    eval_steps=1000,
    save_steps=1000,
    num_train_epochs=30,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",  # Enable wandb reporting
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    run_name=run_name,  # Use the same run name
)

trainer = Trainer(
    model=model.model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Train the model
trainer.train()

# Save the model
model.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print(f"\n✓ Model saved to {OUT_DIR}")

# Log model as artifact
artifact = wandb.Artifact(
    name=f"climb-gpt-model-{run_name}",
    type="model",
    description="Trained KilterGPT model"
)
artifact.add_dir(OUT_DIR)
wandb.log_artifact(artifact)

# Evaluate on test set
test_results = trainer.evaluate(datasets["test"])
print(f"\n✓ Test Loss: {test_results['eval_loss']:.4f}")

# Log test results to wandb
wandb.log({
    "test/loss": test_results['eval_loss'],
})

# Create a summary table
wandb.summary["final_test_loss"] = test_results['eval_loss']
wandb.summary["model_path"] = OUT_DIR

# Finish the wandb run
wandb.finish()

print(f"\n✓ Training complete! View results at: {wandb.run.url}")

/Users/nottreepat/Downloads/KilterTransformer-2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: Currently logged in as: treepatchantaurai (treepatchantaurai-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loaded 76992 routes from cache data/climbs_cleaned.csv
Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('hand1208', 570), ('finish1554', 1751), ('finish1253', 751), ('finish1148', 331), ('feet1449', 1328), ('finish1295', 919), ('finish1564', 1791), ('finish1310', 979), ('feet1088', 88), ('finish1585', 1875)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to models/climb_gpt/run_20251101_165006


In [ ]:
e1e0588a31a395aea6b901f98b9dfbf4ac0cf85a